In [1]:
import pandas as pd
import datetime
from dateutil import parser as dtparser
from IPython.display import display, clear_output

### Importing the modules
import tomtom_api
import processor
import transformer
import csv_handler

In [2]:
# Define a TomTom API Client
ttapi = tomtom_api.Client(api_key = "SdbkkAPVV6GxzS7beuYj8mqYnSRWgUmx")

# Define a tomtom processor
tt_processor = processor.Processor(ttapi)
transformer = transformer.Transformer()
csv_handler = csv_handler.CSVHandler()

In [3]:
# Starting Point and Ending Point of Route Analysis
centroids = csv_handler.readCentroids()
from_cluster = 0
to_cluster = 1
start_point = f"{centroids['Latitudine'].iloc[from_cluster]},{centroids['Longitudine'].iloc[from_cluster]}"              
end_point = f"{centroids['Latitudine'].iloc[to_cluster]},{centroids['Longitudine'].iloc[to_cluster]}"
print (f"Start Point: {start_point}, End Point: {end_point}")

Start Point: 40.592006479897954,17.1154860457833, End Point: 40.59368321711289,17.111410033960354


In [4]:
# Date Range for Route Analysis
year = 2024
month = 7
day_range = range(1, 31)

# define time slots in a day
slots = ["00:00-06:00", "06:00-12:00", "12:00-18:00", "18:00-23:59"]

In [5]:
# Get the route summaries for each day in the date range
display = display("Processing historing summaries...", display_id=True)
for day in day_range:
    date = datetime.datetime(year, month, day)
    route_summaries = tt_processor.getHistoricData(start_point, end_point, date)
    csv_handler.writeHistoricData(route_summaries, date)
    display.update(f"Processing historic summaries { day / len(day_range) * 100 }% ...")
display.update("Historic summaries processed.")

Retrieved data: 100% >> 2024-07-30 23:30:00


In [5]:
# Calculate the traffic data for each day in the date range
dd = display("Calculating traffic data...", display_id=True)
for day in day_range:
    date = datetime.datetime(year, month, day)
    historic_data = csv_handler.readHistoricData(date)
    traffic_data = transformer.calculateTrafficData(historic_data)
    csv_handler.writeTrafficData(traffic_data, date)
    dd.update(f"Calculating traffic data { round(day / len(day_range) * 100) }% ...")
dd.update("Traffic data calculated.")

Computing traffic: 100% >> 2024-07-30T23:30:00+02:00


In [7]:
for day in day_range:
    date = datetime.datetime(year, month, day)
    traffic_data = csv_handler.readTrafficData(date)
    slotted_traffic = transformer.calculateSlottedTraffic(traffic_data, slots)
    csv_handler.writeSlottedTrafficData(slotted_traffic, date)
print ("Done")

Done


In [1]:
combined_data = csv_handler.readAllSlotted()
traffic = transformer.getAggregatedTrafficData(combined_data, slots)

NameError: name 'csv_handler' is not defined

In [5]:
traffic

,slot,avg_traffic_speed,avg_density_factor,avg_number_of_vehicles,points
0,00:00-06:00,37.323478,1.305023,220.709722,"[{'latitude': 41.13407, 'longitude': 16.81843}..."
1,06:00-12:00,39.658330,1.236944,209.140278,"[{'latitude': 41.13407, 'longitude': 16.81843}..."
2,12:00-18:00,32.346674,1.470856,248.780556,"[{'latitude': 41.13407, 'longitude': 16.81843}..."
3,18:00-23:59,34.988625,1.373102,232.279167,"[{'latitude': 41.13407, 'longitude': 16.81843}..."
